In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))



In [ ]:
import torch
from torch import nn

class DiscriminatorCNNVersion1(nn.Module):
    # CNN network that combines temporal data (log spectrogram)
    # and summarized audio data such as MFCCs, zero crossings and so forth at the linear layer.

    def __init__(self, handcrafted_features=68):

        super().__init__()  # inherit nn module

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1)  # first layer convolution

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(kernel_size=2)  # reduce dimensionality

        # to further reduce the size of the huge vector
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.flatten = nn.Flatten(start_dim=1)

        # input is 16, 4x4, feature maps for each input
        self.hidden = nn.Linear(16*4*4 + handcrafted_features, 64)

        # output of hidden is.. a 64 dim vector, we output a logit!
        self.out = nn.Linear(64, 1)

    def forward(self, logSpec, compressedFeatures):
        # input -> conv -> activations -> pool -> reduce adaptive pool -> flatten  -> hidden layer -> output
        x = self.conv1(logSpec)
        x = self.relu(x)
        x = self.pool(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)  # (Batch size, 256)

        # combine x and compressed non-temporals
        x = self.hidden(torch.cat([x, compressedFeatures], dim=1))
        x = self.relu(x)  # hidden layer stuff goes through relu again

        return self.out(x)